### RAG Pipeline - Data Ingestion ti Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\arthika.l\AppData\Local\Temp\ipykernel_30744\885881214.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\arthika.l\OneDrive - ascendion\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path

### Read all the pdf inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    ## find all the pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add socurce information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal documents loader: {len(all_documents)}")
    return all_documents
#Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

    

found 1 PDF files to process

Processing: Workflow_output.pdf
Loaded 4 pages

Total documents loader: 4


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'jsPDF 3.0.1', 'creator': 'PyPDF', 'creationdate': '2025-12-10T13:04:05+05:30', 'source': '..\\data\\pdf\\Workflow_output.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Workflow_output.pdf', 'file_type': 'pdf'}, page_content='Description:\nINPUT: take the source file\nThe Log Analyzer agent processes raw logs and converts them into structured insights by following a defined\nalgorithmic workflow.\nBEGIN\n1. INPUT_HANDLING:\n   IF input is plain text OR JSON OR .log OR .txt OR .csv OR .zip THEN\n       Load input\n   ELSE IF input is streaming chunks THEN\n       Append chunks to buffer\n   ENDIF\n   Normalize line breaks and encoding.\n2. PARSE_AND_NORMALIZE:\n   FOR each log entry IN input:\n       Detect format:\n\x00 \x00 \x00 \x00 \x00 \x00 \x00 \x00 \x00 \x00 \x00 \x00I\x00F\x00 \x00J\x00S\x00O\x00N\x00 !’\x00 \x00p\x00a\x00r\x00s\x00e\x00 \x00J\x00S\x00O\x00N\x00 \x00f\x00i\x00e\x00l\x00d\x00s\n\x00 \x00 \x00 \x00 \x00 \x00 \

In [4]:
### Chunking

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks


Split 4 documents into 9 chunks

Example chunk:
Content: Description:
INPUT: take the source file
The Log Analyzer agent processes raw logs and converts them into structured insights by following a defined
algorithmic workflow.
BEGIN
1. INPUT_HANDLING:
   I...
Metadata: {'producer': 'jsPDF 3.0.1', 'creator': 'PyPDF', 'creationdate': '2025-12-10T13:04:05+05:30', 'source': '..\\data\\pdf\\Workflow_output.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Workflow_output.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'jsPDF 3.0.1', 'creator': 'PyPDF', 'creationdate': '2025-12-10T13:04:05+05:30', 'source': '..\\data\\pdf\\Workflow_output.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Workflow_output.pdf', 'file_type': 'pdf'}, page_content='Description:\nINPUT: take the source file\nThe Log Analyzer agent processes raw logs and converts them into structured insights by following a defined\nalgorithmic workflow.\nBEGIN\n1. INPUT_HANDLING:\n   IF input is plain text OR JSON OR .log OR .txt OR .csv OR .zip THEN\n       Load input\n   ELSE IF input is streaming chunks THEN\n       Append chunks to buffer\n   ENDIF\n   Normalize line breaks and encoding.\n2. PARSE_AND_NORMALIZE:\n   FOR each log entry IN input:\n       Detect format:\n\x00 \x00 \x00 \x00 \x00 \x00 \x00 \x00 \x00 \x00 \x00 \x00I\x00F\x00 \x00J\x00S\x00O\x00N\x00 !’\x00 \x00p\x00a\x00r\x00s\x00e\x00 \x00J\x00S\x00O\x00N\x00 \x00f\x00i\x00e\x00l\x00d\x00s\n\x00 \x00 \x00 \x00 \x00 \x00 \

Embedding And VectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [7]:
class EmbeddingManager:
    """Handles docment embedding generation using SentenceTransfer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the EmbeddingManager with a specified model.
        Args:
            model_name (str): The name of the SentenceTransformer model to use.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
   
    def _load_model(self):
        """Load the SentenceTransfer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts :List of strings to generate embeddings for.

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    


In [8]:
    ## initialize the embedding manager

    embedding_manager = EmbeddingManager()
    embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6308.51it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\arthika.l\AppData\Local\Temp\ipykernel_30744\303904968.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [10]:
## Vector Store


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
            """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}

            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embedding for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
     
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore




Vector store initialized. Collection: pdf_documents
Existing documents in collection: 18


In [11]:

chunks


[Document(metadata={'producer': 'jsPDF 3.0.1', 'creator': 'PyPDF', 'creationdate': '2025-12-10T13:04:05+05:30', 'source': '..\\data\\pdf\\Workflow_output.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Workflow_output.pdf', 'file_type': 'pdf'}, page_content='Description:\nINPUT: take the source file\nThe Log Analyzer agent processes raw logs and converts them into structured insights by following a defined\nalgorithmic workflow.\nBEGIN\n1. INPUT_HANDLING:\n   IF input is plain text OR JSON OR .log OR .txt OR .csv OR .zip THEN\n       Load input\n   ELSE IF input is streaming chunks THEN\n       Append chunks to buffer\n   ENDIF\n   Normalize line breaks and encoding.\n2. PARSE_AND_NORMALIZE:\n   FOR each log entry IN input:\n       Detect format:\n\x00 \x00 \x00 \x00 \x00 \x00 \x00 \x00 \x00 \x00 \x00 \x00I\x00F\x00 \x00J\x00S\x00O\x00N\x00 !’\x00 \x00p\x00a\x00r\x00s\x00e\x00 \x00J\x00S\x00O\x00N\x00 \x00f\x00i\x00e\x00l\x00d\x00s\n\x00 \x00 \x00 \x00 \x00 \x00 \

In [12]:
## Convert the text to embeddings

texts=[doc.page_content for doc in chunks]


## Generate the embeddings

embeddings=embedding_manager.generate_embeddings(texts)

## store in vector database

vectorstore.add_documents(chunks,embeddings)


Generating embeddings for 9 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.48it/s]

Generated embeddings with shape: (9, 384)
Adding 9 documents to vector store...
Successfully added 9 documents to vector store
Total documents in collection: 27


In [13]:
from typing import List, Any

## Retriever pipeline from vectorstore

class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        max_distance: float = None
    ) -> List[dict[str, Any]]:
        """
        Retrieve relevant documents for a query.

        Args:
            query: User query
            top_k: Number of documents to retrieve
            max_distance: Optional distance threshold.
                          Smaller distance = better match.

        Returns:
            List of retrieved documents.
        """

        print("=" * 60)
        print(f"Retrieving documents for query: {query}")
        print("=" * 60)

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        print(f"Embedding shape: {query_embedding.shape}")

        try:
            # Query ChromaDB
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            print("\nRetrieved from ChromaDB")
            print("-----------------------")
            print("IDs:", results["ids"])
            print("Distances:", results["distances"])

            retrieved_docs = []

            if results["documents"] and results["documents"][0]:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for rank, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances),
                    start=1
                ):

                    print(f"\nResult #{rank}")
                    print(f"Document ID : {doc_id}")
                    print(f"Distance    : {distance}")

                    # Optional distance filtering
                    if max_distance is not None and distance > max_distance:
                        print("Skipped (distance too large)")
                        continue

                    retrieved_docs.append({
                        "id": doc_id,
                        "content": document,
                        "metadata": metadata,
                        "distance": distance,
                        "rank": rank
                    })

            print("\n===================================")
            print(f"Retrieved {len(retrieved_docs)} documents")
            print("===================================\n")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


# Create retriever
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [14]:
rag_retriever

In [15]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: What is attention is all you need
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 61.47it/s]

Generated embeddings with shape: (1, 384)
Embedding shape: (384,)

Retrieved from ChromaDB
-----------------------
IDs: [['doc_f366d4e1_2', 'doc_4ab16a80_2', 'doc_f9701986_2', 'doc_a6afad8e_0', 'doc_212d7708_0']]
Distances: [[1.6107866764068604, 1.6107866764068604, 1.6107866764068604, 1.679671287536621, 1.679671287536621]]

Result #1
Document ID : doc_f366d4e1_2
Distance    : 1.6107866764068604

Result #2
Document ID : doc_4ab16a80_2
Distance    : 1.6107866764068604

Result #3
Document ID : doc_f9701986_2
Distance    : 1.6107866764068604

Result #4
Document ID : doc_a6afad8e_0
Distance    : 1.679671287536621

Result #5
Document ID : doc_212d7708_0
Distance    : 1.679671287536621

Retrieved 5 documents



[{'id': 'doc_f366d4e1_2',
  'content': 'IF resource-related patterns detected (Memory, CPU, Throttle) THEN\n       Mark anomaly("Resource Issue")\n   ENDIF\n5. ROOT_CAUSE_ANALYSIS:\n   FOR each critical_event:\n       IF "Caused by" is present THEN\n           Link event to cause\n       ELSE IF event closely precedes other failures THEN\n           Infer cause-effect chain\n       ENDIF\n   ENDFOR\n6. PERFORMANCE_ANALYSIS:\n   IF response_time > threshold THEN\n       Add to slow_requests\n   ENDIF\n   IF timestamps show delay patterns THEN\n       Flag latency issue\n   ENDIF\n   IF endpoints repeatedly slow THEN\n       Mark bottleneck\n   ENDIF\n7. SUMMARIZE_AND_REPORT:\n   Generate:\n       Summary = overall system issue in 1–2 lines\n       Key_Events = all high-impact log entries\n       Root_Cause = most likely triggering factor\n       Anomalies = unusual or repeated patterns\n       Performance_Findings = latency, bottlenecks\n       Security_Concerns = authentication/authori

In [16]:

rag_retriever.retrieve("Unified Multi-task Learning Framework")

Retrieving documents for query: Unified Multi-task Learning Framework
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 67.80it/s]

Generated embeddings with shape: (1, 384)
Embedding shape: (384,)

Retrieved from ChromaDB
-----------------------
IDs: [['doc_45a1dcb8_3', 'doc_f735a015_3', 'doc_5388ae10_3', 'doc_a6afad8e_0', 'doc_05f47c93_0']]
Distances: [[1.5610277652740479, 1.5610277652740479, 1.5610277652740479, 1.7462027072906494, 1.7462027072906494]]

Result #1
Document ID : doc_45a1dcb8_3
Distance    : 1.5610277652740479

Result #2
Document ID : doc_f735a015_3
Distance    : 1.5610277652740479

Result #3
Document ID : doc_5388ae10_3
Distance    : 1.5610277652740479

Result #4
Document ID : doc_a6afad8e_0
Distance    : 1.7462027072906494

Result #5
Document ID : doc_05f47c93_0
Distance    : 1.7462027072906494

Retrieved 5 documents



[{'id': 'doc_45a1dcb8_3',
  'content': 'Performance_Findings = latency, bottlenecks\n       Security_Concerns = authentication/authorization failures\n       Recommendations = clear troubleshooting steps\n       Severity = (Critical / High / Medium / Low)\n       Timeline = ordered events by timestamp\nRETURN all structured fields.\nEND\nYou have access to the KnowledgeRAGTool. It is advised to use this tool when you have access to it irrespective of\nwhether a knowledge retrieval task has been assigned explicitly.',
  'metadata': {'creationdate': '2025-12-10T13:04:05+05:30',
   'source': '..\\data\\pdf\\Workflow_output.pdf',
   'source_file': 'Workflow_output.pdf',
   'total_pages': 4,
   'content_length': 472,
   'creator': 'PyPDF',
   'page_label': '2',
   'page': 1,
   'doc_index': 3,
   'producer': 'jsPDF 3.0.1',
   'file_type': 'pdf'},
  'distance': 1.5610277652740479,
  'rank': 1},
 {'id': 'doc_f735a015_3',
  'content': 'Performance_Findings = latency, bottlenecks\n       Securi

In [1]:
## RAG Pipeline - VectorDB to LLM output Generation

import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found!")

print("Groq API key loaded successfully!")

Groq API key loaded successfully!


In [18]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="gemma2-9b-it",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [19]:
answer=rag_simple("llm=what is attention mechanism?", rag_retriever,llm)
print(answer)

Retrieving documents for query: llm=what is attention mechanism?
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 96.91it/s]

Generated embeddings with shape: (1, 384)
Embedding shape: (384,)

Retrieved from ChromaDB
-----------------------
IDs: [['doc_45a1dcb8_3', 'doc_f735a015_3', 'doc_5388ae10_3']]
Distances: [[1.7606745958328247, 1.7606745958328247, 1.7606745958328247]]

Result #1
Document ID : doc_45a1dcb8_3
Distance    : 1.7606745958328247

Result #2
Document ID : doc_f735a015_3
Distance    : 1.7606745958328247

Result #3
Document ID : doc_5388ae10_3
Distance    : 1.7606745958328247

Retrieved 3 documents



APIConnectionError: Connection error.

In [21]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])



TypeError: RAGRetriever.retrieve() got an unexpected keyword argument 'score_threshold'

In [22]:
# Advanced RAG Pipeline: Streaming, Citations, History, Summarization

from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:

        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources == []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]

            prompt = f"""Use the following context to answer the question concisely. \nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.formate(context=context, question=question)])
            answer = response.content

        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations)if citations else answer

        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("summary:", result['summary'])
print("History:", result['history'][-1])

TypeError: RAGRetriever.retrieve() got an unexpected keyword argument 'score_threshold'